# Module 5: Critic-Refiner — Production Deployment

Deploys two separate A2A specialist runtimes:
- **Writer** — Generator: produces or revises the memo
- **Critic** — Evaluates the memo: `APPROVED` or `REVISION NEEDED: ...`

`chain.py` coordinates the loop locally — passing context explicitly in each A2A call.

---

## Step 1: Install dependencies

In [ ]:
%pip install "strands-agents[a2a]>=1.52.0" bedrock-agentcore boto3 httpx

# ⚠️  If this cell installs new packages (e.g. a2a-sdk), restart the kernel
# before continuing: Kernel → Restart Kernel, then re-run from Step 2 onward.
# You do NOT need to re-deploy — the ARNs in .env_arns are still valid.

---

## Step 2: Set up execution roles

Creates the IAM execution roles for the AgentCore runtimes (idempotent — safe to re-run).

In [ ]:
import sys, os, boto3
sys.path.insert(0, "../../shared")
import deploy_utils as u

session = u.get_session()
account = u.get_account(session)
bucket  = u.code_bucket_name(account, u.REGION)
iam     = session.client("iam", region_name=u.REGION)

runtime_role_arn = u.ensure_runtime_role(
    iam, f"workshop-agentcore-m5-runtime-role",
    account, u.REGION, bucket,
)
os.environ["AGENTCORE_RUNTIME_ROLE_ARN"] = runtime_role_arn
print(f"Runtime role: {runtime_role_arn}")

---

## Step 3: Deploy

Deploys the **Writer** and **Critic** runtimes in parallel (~3-5 min).  
`chain.py` manages the Generator↔Critic loop by passing context explicitly between them.

In [ ]:
!python deploy.py --name-prefix m5

In [ ]:
import os
from pathlib import Path

_candidates = [
    Path("/workshop/samples/05-critic-refiner/production/.env_arns"),
    Path(__file__).parent / ".env_arns" if "__file__" in dir() else None,
    Path(".env_arns"),
]
_env_path = next((p for p in _candidates if p and p.exists()), None)
if _env_path is None:
    raise FileNotFoundError("Cannot find .env_arns — run the deploy step first.")

with open(_env_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line.startswith("export "):
            key, _, val = line[len("export "):].partition("=")
            os.environ[key.strip()] = val.strip()

WRITER_ARN = os.environ["WRITER_RUNTIME_ARN"]
CRITIC_ARN = os.environ["CRITIC_RUNTIME_ARN"]

print(f"Reading from: {_env_path}")
print(f"Writer ARN:   {WRITER_ARN}")
print(f"Critic ARN:   {CRITIC_ARN}")

---

## Step 4: Run the chain

`chain.py` calls the Writer and Critic runtimes alternately until the Critic says `APPROVED`.  
Context — brief, draft, feedback — is passed explicitly in each A2A call.

In [ ]:
import sys, importlib, os

sys.path.insert(0, ".")

# Force-reload chain.py so it picks up the latest ARNs from .env_arns
import chain as _chain_mod
importlib.reload(_chain_mod)
from chain import run_chain

BRIEF = (
    "NovaCart Premium Tier: Options A ($19.99/mo invite-only), "
    "B ($14.99/mo 5% pilot), C ($12.99/mo full launch). "
    "Target: +15% CLV in 6 months. Budget: $2M."
)

print(f"Using Writer ARN: {os.environ.get('WRITER_RUNTIME_ARN', 'NOT SET')}")
print("Running Critic-Refiner (2 separate runtimes: Writer ↔ Critic)...")
print("─" * 60)
print(run_chain(BRIEF))

---

## 💻 Multi-turn interactive chat

Run in a terminal for an interactive session (submit multiple briefs):

```bash
cd samples/05-critic-refiner/production
source .env_arns
python chat.py
```

Type a brief and press Enter. Press Enter on an empty line or type `quit` to exit.

---

## Step 5: Observability

After running `chain.py`, traces appear in **CloudWatch > X-Ray > Traces** or **Amazon Bedrock > AgentCore > Observability**.

What you see per invocation:
- One Writer span per draft (cycle 1: initial draft, cycle 2+: revisions)
- One Critic span per evaluation
- Total cycles = number of Writer→Critic iterations until APPROVED

No extra configuration needed: `aws-opentelemetry-distro` is in `requirements.txt` and AgentCore instruments both specialists automatically.

---

## Step 6: Cleanup

Uncomment and run the cell below to delete all AWS resources created by this module.

In [ ]:
# Uncomment and run to delete all resources created by this module.

# !python cleanup.py --name-prefix m5

# Verify:
# import boto3, os
# REGION = os.environ.get("AWS_REGION", "us-east-1")
# remaining = boto3.client("bedrock-agentcore-control", region_name=REGION).list_agent_runtimes()
# print([rt["agentRuntimeName"] for rt in remaining.get("agentRuntimes", [])])